# Task 1 - Simple RAG

In [ ]:
import cv2
import os
import torch
from transformers import AutoProcessor, AutoModel, AutoTokenizer
from pathlib import Path

### 1.1 Extract Keyframes
Using OpenCV, i can extract every 12th frame of the video

In [ ]:
# load the video
video = cv2.VideoCapture("sampleClip.mp4")

# video.get(cv2.CAP_PROP_FPS)

In [ ]:
os.makedirs("frames", exist_ok=True)

i = 0
while True:
    ret, frame = video.read()

    if not ret:
        break

    # extract the 12th frame
    if i%12 == 0:
        cv2.imwrite(f"frames\\frame_{i:06d}.jpg", frame)

    i+=1

### 1.2 Encode video using CLIP

In [ ]:
model_id = "google/siglip2-so400m-patch14-384"

processor = AutoProcessor.from_pretrained(model_id)
model = AutoModel.from_pretrained(model_id)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

In [ ]:
query = ["Effel tower"]

inputs = processor(query, padding = True, return_tensors="pt")
inputs.to(device)

# image = cv2.imread("frames//frame_000000.jpg")
# new_image = processor(image, return_tensors="pt")

with torch.no_grad():
    text_features = model.get_text_features(**inputs)

# Normalize embeddings for cosine similarity
text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"


import torch

from transformers import AutoModel, AutoProcessor

model_id = "google/siglip2-so400m-patch14-384"

processor = AutoProcessor.from_pretrained(model_id)

model = AutoModel.from_pretrained(
    model_id,
    device_map="auto"
).eval()

texts = [
    "HOME084 Timbangan Badan Digital Kaca Transparan 28CM Body Scale Personal Scale",
    "26cm Timbangan Badan digital personal scale weight",
    "33cm Timbangan Badan digital personal scale weight",
]

# NOTE: lowercasing and padding/truncation to length 64 are applied automatically by the processor pipeline.
inputs = processor(
    text=texts,
    padding=True,
    truncation=True,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    text_features = model.get_text_features(**inputs)

# Normalize embeddings for cosine similarity
text_features = text_features / text_features.norm(p=2, dim=-1, keepdim=True)

Loading weights: 100%|██████████| 888/888 [00:05<00:00, 164.09it/s]
Some parameters are on the meta device because they were offloaded to the cpu.


AttributeError: 'BaseModelOutputWithPooling' object has no attribute 'norm'